# Stress Prediction — 전처리 재점검 (EDA)

지금까지 모델/피처만 계속 바꿨지, 데이터 자체를 제대로 본 적은 없었습니다. 확인할 것:

1. 결측치 패턴 (컬럼별, 행별)
2. 이상치 (IQR 기준)
3. Train/Test 분포 차이 (컬럼별 평균 비교)
4. 타겟(stress_score) 분포
5. **Train-Test 근접/완전 중복 샘플 존재 여부** — 이게 왜 v34의 1-NN 방식이 잘 먹혔는지 설명해줄 수 있음
6. 중복 행(row) 자체가 train 안에 있는지


In [1]:

from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd

TARGET = "stress_score"
ID_COLUMN = "ID"

DATA_DIR = Path("/Users/bitsaem/Desktop/stress_project/open")

train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")

print("train shape:", train.shape)
print("test shape:", test.shape)
print()
print(train.dtypes)


train shape: (3000, 18)
test shape: (3000, 17)

ID                           object
gender                       object
age                           int64
height                      float64
weight                      float64
cholesterol                 float64
systolic_blood_pressure       int64
diastolic_blood_pressure      int64
glucose                     float64
bone_density                float64
activity                     object
smoke_status                 object
medical_history              object
family_medical_history       object
sleep_pattern                object
edu_level                    object
mean_working                float64
stress_score                float64
dtype: object


## 1. 결측치 패턴

In [2]:

missing_train = train.isna().sum().sort_values(ascending=False)
missing_test = test.isna().sum().sort_values(ascending=False)

print("### Train 결측치 (컬럼별, 비율%)")
display((missing_train[missing_train > 0] / len(train) * 100).round(2))

print()
print("### Test 결측치 (컬럼별, 비율%)")
display((missing_test[missing_test > 0] / len(test) * 100).round(2))

print()
print("### 행(row) 단위 결측 개수 분포 (train)")
row_missing = train.isna().sum(axis=1)
display(row_missing.value_counts().sort_index())


### Train 결측치 (컬럼별, 비율%)


family_medical_history    49.53
medical_history           42.97
mean_working              34.40
edu_level                 20.23
dtype: float64


### Test 결측치 (컬럼별, 비율%)


family_medical_history    47.20
medical_history           43.63
mean_working              33.60
edu_level                 21.57
dtype: float64


### 행(row) 단위 결측 개수 분포 (train)


0    690
1    815
2    964
3    453
4     78
Name: count, dtype: int64

## 2. 이상치 (IQR 기준, 수치형 컬럼)

In [3]:

numeric_cols = train.select_dtypes(include=["number"]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c not in (ID_COLUMN, TARGET)]

outlier_rows = []
for col in numeric_cols:
    q1, q3 = train[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_outliers = ((train[col] < lower) | (train[col] > upper)).sum()
    outlier_rows.append({
        "column": col, "n_outliers": n_outliers,
        "pct": round(n_outliers / len(train) * 100, 2),
        "valid_range": f"[{lower:.1f}, {upper:.1f}]",
        "actual_range": f"[{train[col].min():.1f}, {train[col].max():.1f}]",
    })

outlier_df = pd.DataFrame(outlier_rows).sort_values("n_outliers", ascending=False)
display(outlier_df)


,column,n_outliers,pct,valid_range,actual_range
8,mean_working,33,1.10,"[5.0, 13.0]","[4.0, 16.0]"
5,diastolic_blood_pressure,19,0.63,"[69.5, 121.5]","[60.0, 124.0]"
3,cholesterol,17,0.57,"[166.8, 300.6]","[148.8, 313.4]"
6,glucose,13,0.43,"[76.0, 177.1]","[70.0, 185.7]"
2,weight,2,0.07,"[34.8, 110.0]","[36.3, 120.9]"
4,systolic_blood_pressure,2,0.07,"[100.5, 192.5]","[97.0, 191.0]"
1,height,1,0.03,"[141.5, 195.7]","[141.1, 195.3]"
0,age,0,0.00,"[-20.5, 127.5]","[17.0, 89.0]"
7,bone_density,0,0.00,"[-0.5, 2.4]","[-0.2, 2.0]"


## 3. Train vs Test 분포 차이 (수치형 컬럼 평균/표준편차 비교)

차이가 크면(예: 평균이 표준편차의 상당 부분만큼 차이나면) train으로 학습한 모델이
test에서 체계적으로 편향될 수 있습니다.


In [4]:

compare_rows = []
for col in numeric_cols:
    train_mean, train_std = train[col].mean(), train[col].std()
    test_mean, test_std = test[col].mean(), test[col].std()
    diff_in_std = abs(train_mean - test_mean) / (train_std + 1e-9)
    compare_rows.append({
        "column": col,
        "train_mean": round(train_mean, 3), "test_mean": round(test_mean, 3),
        "train_std": round(train_std, 3), "test_std": round(test_std, 3),
        "mean_diff_in_std_units": round(diff_in_std, 3),
    })

compare_df = pd.DataFrame(compare_rows).sort_values("mean_diff_in_std_units", ascending=False)
display(compare_df)
print()
print("mean_diff_in_std_units가 0.1 이상이면 분포 차이가 신경쓰일 만한 수준입니다.")


,column,train_mean,test_mean,train_std,test_std,mean_diff_in_std_units
8,mean_working,8.717,8.844,1.629,1.621,0.078
6,glucose,126.468,126.949,18.537,17.904,0.026
0,age,53.068,53.398,20.672,20.286,0.016
3,cholesterol,233.911,234.296,24.333,24.704,0.016
4,systolic_blood_pressure,146.144,145.913,15.845,16.104,0.015
2,weight,72.487,72.667,13.167,13.211,0.014
5,diastolic_blood_pressure,95.504,95.637,9.895,9.978,0.013
1,height,168.586,168.652,9.352,9.235,0.007
7,bone_density,0.933,0.933,0.445,0.442,0.000



mean_diff_in_std_units가 0.1 이상이면 분포 차이가 신경쓰일 만한 수준입니다.


## 4. 타겟(stress_score) 분포

In [5]:

print(train[TARGET].describe())
print()
print("0.5 이하 비율:", (train[TARGET] <= 0.5).mean())
print("고유값 개수:", train[TARGET].nunique(), "/ 전체", len(train))
print()
print("값 분포 (상위 20개 빈도):")
display(train[TARGET].value_counts().head(20))


count    3000.000000
mean        0.482130
std         0.288252
min         0.000000
25%         0.230000
50%         0.480000
75%         0.730000
max         1.000000
Name: stress_score, dtype: float64

0.5 이하 비율: 0.526
고유값 개수: 101 / 전체 3000

값 분포 (상위 20개 빈도):


stress_score
0.12    44
0.54    43
0.71    42
0.74    41
0.37    40
0.19    40
0.16    40
0.59    40
0.40    39
0.13    39
0.23    38
0.55    37
0.26    37
0.21    37
0.02    36
0.17    36
0.04    36
0.15    35
0.72    35
0.52    35
Name: count, dtype: int64

## 5. Train-Test 근접/완전 중복 샘플 확인 (핵심)

수치형 컬럼 기준으로 test의 각 샘플과 가장 가까운 train 샘플의 거리를 계산합니다.
거리가 0에 가까운(=사실상 동일한) 샘플이 많으면, 1-NN 방식이 왜 그렇게 잘 먹혔는지 설명됩니다.


In [6]:

from sklearn.preprocessing import StandardScaler
from scipy.spatial import cKDTree

scaler = StandardScaler()
train_numeric = train[numeric_cols].fillna(train[numeric_cols].median())
test_numeric = test[numeric_cols].fillna(train[numeric_cols].median())

train_scaled = scaler.fit_transform(train_numeric)
test_scaled = scaler.transform(test_numeric)

tree = cKDTree(train_scaled)
nearest_dist, nearest_idx = tree.query(test_scaled, k=1)

print("test 샘플 각각의 train 내 최근접 거리(표준화 유클리드) 분포:")
print(pd.Series(nearest_dist).describe())
print()
print("거리 0.1 미만(사실상 거의 동일)인 test 샘플 개수:", (nearest_dist < 0.1).sum(), "/", len(test))
print("거리 0.5 미만인 test 샘플 개수:", (nearest_dist < 0.5).sum(), "/", len(test))
print()

very_close = nearest_dist < 0.1
if very_close.sum() > 0:
    print("### 거리가 매우 가까운 test-train 쌍 예시 (최대 10개)")
    close_test_idx = np.where(very_close)[0][:10]
    for ti in close_test_idx:
        matched_train_idx = nearest_idx[ti]
        print(f"test row {ti} <-> train row {matched_train_idx}, distance={nearest_dist[ti]:.4f}")
    display(test.iloc[close_test_idx[:5]][numeric_cols])
    display(train.iloc[nearest_idx[close_test_idx[:5]]][numeric_cols + [TARGET]])


test 샘플 각각의 train 내 최근접 거리(표준화 유클리드) 분포:
count    3000.000000
mean        0.889012
std         0.514395
min         0.000000
25%         0.702353
50%         0.968189
75%         1.234147
max         2.991706
dtype: float64

거리 0.1 미만(사실상 거의 동일)인 test 샘플 개수: 383 / 3000
거리 0.5 미만인 test 샘플 개수: 672 / 3000

### 거리가 매우 가까운 test-train 쌍 예시 (최대 10개)
test row 12 <-> train row 2237, distance=0.0542
test row 17 <-> train row 515, distance=0.0484
test row 22 <-> train row 250, distance=0.0802
test row 32 <-> train row 1286, distance=0.0973
test row 42 <-> train row 1959, distance=0.0198
test row 51 <-> train row 2937, distance=0.0000
test row 52 <-> train row 1750, distance=0.0560
test row 69 <-> train row 153, distance=0.0646
test row 73 <-> train row 127, distance=0.0553
test row 77 <-> train row 2320, distance=0.0268


,age,height,weight,cholesterol,systolic_blood_pressure,diastolic_blood_pressure,glucose,bone_density,mean_working
12,77,166.95,63.75,290.65,164,88,130.10,0.50,NaN
17,56,165.12,64.09,221.19,163,97,115.20,0.84,8.0
22,58,176.12,78.55,242.87,140,106,91.20,0.65,11.0
32,79,174.14,87.36,219.65,187,99,134.68,0.87,NaN
42,76,165.30,61.99,255.13,155,107,139.47,0.34,NaN


,age,height,weight,cholesterol,systolic_blood_pressure,diastolic_blood_pressure,glucose,bone_density,mean_working,stress_score
2237,76,167.04,63.75,290.65,164,88,130.10,0.49,NaN,0.69
515,55,165.12,64.09,221.19,163,97,115.20,0.84,8.0,0.20
250,57,176.12,78.55,242.87,141,106,91.39,0.65,11.0,0.52
1286,81,174.14,87.23,219.65,187,99,134.68,0.87,NaN,0.37
1959,76,165.39,62.12,255.38,155,107,139.65,0.34,NaN,0.87


## 6. Train 내부 중복/근접 중복 행 확인

In [7]:

duplicate_rows = train[numeric_cols].duplicated().sum()
print("train 내 완전 중복 행(수치형 기준) 개수:", duplicate_rows)

# 같은 입력값인데 타겟(stress_score)이 다른 경우 (라벨 노이즈 의심)
dup_mask = train[numeric_cols].duplicated(keep=False)
if dup_mask.sum() > 0:
    dup_groups = train[dup_mask].groupby(numeric_cols)[TARGET].agg(["nunique", "mean", "std", "count"])
    inconsistent = dup_groups[dup_groups["nunique"] > 1]
    print()
    print("같은 입력인데 stress_score가 다른 그룹 수:", len(inconsistent))
    if len(inconsistent) > 0:
        display(inconsistent.head(10))


train 내 완전 중복 행(수치형 기준) 개수: 6

같은 입력인데 stress_score가 다른 그룹 수: 0
